# API-232 Lab (SOLUTIONS): Do NFL Teams Fall for the Sunk-Cost Fallacy?
### Confounding, double ML, sensitivity analysis and regression discontinuity with NFL draft data

Every spring, NFL teams take turns choosing college players in the **draft**. First-round picks are the most expensive: they get the biggest guaranteed contracts, and the team spends its most valuable selection on them.

Economics says **past costs shouldn't affect current decisions**. Once a player is on the roster, the coach should play whoever gives the team the best chance to win, whatever the player cost. If teams give first-round picks *more playing time than equally good later picks*, they are falling for the **sunk-cost fallacy**.

This matters well beyond football. Governments keep funding failing programmes, weapons systems and infrastructure projects "because we've already spent so much". Measuring the sunk-cost effect in the field is hard for the same reason it's hard here: **the projects (or players) that cost the most were usually expected to be the best.**

**The question:** among wide receivers, running backs and tight ends who performed equally well last season, do first-round picks get more playing time than second- and third-round picks?

**Concepts from Assignment 5:**

| Part | Concept | Assignment link |
|---|---|---|
| 1–2 | Confounding; adding controls; bad controls | Problem 2 (LaLonde), Problem 3 |
| 3 | Double machine learning | Problem 2 (`dml_buggy.py`) |
| 4 | Sensitivity analysis | Lecture on omitted-variable bias |
| 5 | Selection on an outcome | Problem 3 (birth-weight paradox) |
| 6 | Regression discontinuity | Problem 4 (tariffs) |

**Time:** about 75 minutes. Answer the written questions in the markdown cells marked **✏️ Your answer**.


## 0. Setup (about 3 minutes)

Run the next cell. It installs `nflreadpy`, the Python package for the free [nflverse](https://github.com/nflverse) NFL data, and downloads three tables:

* **Draft picks:** every player drafted, with round, pick number, position and age.
* **Snap counts** (2013–2025): how many plays each player was on the field for, every game. A play is a "snap".
* **Season statistics** (2013–2025): targets, catches, yards, touchdowns and so on.

It takes about a minute.

In [ ]:
!pip install -q nflreadpy

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
import nflreadpy as nfl

pd.set_option("display.precision", 3)
SEASONS = list(range(2013, 2026))   # snap counts start in 2013

draft = nfl.load_draft_picks().to_pandas()
snaps = nfl.load_snap_counts(SEASONS).to_pandas()
snaps = snaps[snaps.game_type == "REG"]
stats = nfl.load_player_stats(SEASONS, summary_level="reg").to_pandas()
schedules = nfl.load_schedules().to_pandas()
print(f"{len(draft):,} draft picks, {len(snaps):,} player-game snap records, {len(stats):,} player-seasons of stats")

### Building the analysis data (provided)

The next cell builds a **player-season panel**. Read the comments, but you don't need to change anything.

* **Players:** wide receivers (WR), running backs (RB) and tight ends (TE) drafted in **rounds 1–3** from 2013 to 2022. These positions have good performance statistics.
* **Seasons:** each player's **2nd, 3rd and 4th seasons**. We skip the rookie season so that every row has a *previous* season of performance to control for.
* **Outcome, `off_share`:** the player's offensive snaps that season, divided by a typical team's offensive snaps that season. 1.0 means he played every snap; a player who wasn't in the league gets 0.
* **Treatment, `first_round`:** 1 if drafted in round 1, 0 if round 2 or 3.
* **Controls (all from the *previous* season, prefixed `prev_`):** snap share, games, targets, catches, yards, touchdowns, fumbles, efficiency per target and per carry, and EPA per touch. *EPA (expected points added) measures how much each play changed the team's expected score.*

In [ ]:
# --- League-average team snaps per season (denominator for snap shares)
team_games = (snaps.groupby(["season", "team", "game_id"])
              .agg(off=("offense_snaps", "max"), de=("defense_snaps", "max")).reset_index())
team_season = team_games.groupby(["season", "team"])[["off", "de"]].sum().groupby("season").mean().reset_index()

player_season = (snaps.groupby(["pfr_player_id", "season"])[["offense_snaps", "defense_snaps"]]
                 .sum().reset_index().merge(team_season, on="season"))
player_season["off_share"] = (player_season.offense_snaps / player_season.off).clip(upper=1)
player_season["play_share"] = np.maximum(player_season.offense_snaps / player_season.off,
                                         player_season.defense_snaps / player_season.de).clip(upper=1)

# --- Season statistics (summed if a player had rows for two teams)
STATS = ["games", "targets", "receptions", "receiving_yards", "receiving_tds", "receiving_epa",
         "carries", "rushing_yards", "rushing_tds", "rushing_epa", "fumbles_lost_total"]
season_stats = (stats.rename(columns={"player_id": "gsis_id", "season": "year"})
                .groupby(["gsis_id", "year"])[STATS].sum().reset_index())

# --- One row per player per career year (1-4), including years with no snaps
picks = draft[draft.season.between(2013, 2022) & (draft["round"] <= 3)
              & draft.position.isin(["WR", "RB", "TE"])].copy()
rows = []
for k in range(1, 5):
    r = picks[["season", "pfr_player_id", "gsis_id", "pfr_player_name", "round", "pick", "position", "age"]].copy()
    r["career_year"], r["year"] = k, r.season + k - 1
    rows.append(r)
panel = (pd.concat(rows).rename(columns={"season": "draft_year"})
         .merge(player_season[["pfr_player_id", "season", "off_share"]].rename(columns={"season": "year"}),
                on=["pfr_player_id", "year"], how="left")
         .merge(season_stats, on=["gsis_id", "year"], how="left"))
panel[["off_share"] + STATS] = panel[["off_share"] + STATS].fillna(0)   # not in the league = 0

# --- Previous-season versions of every performance variable
panel = panel.sort_values(["pfr_player_id", "career_year"])
prev = panel.groupby("pfr_player_id")[["off_share"] + STATS].shift(1).add_prefix("prev_")
panel = pd.concat([panel, prev], axis=1)
panel["prev_yds_per_target"] = np.where(panel.prev_targets > 0, panel.prev_receiving_yards / panel.prev_targets.clip(lower=1), 0)
panel["prev_yds_per_carry"] = np.where(panel.prev_carries > 0, panel.prev_rushing_yards / panel.prev_carries.clip(lower=1), 0)
panel["prev_epa_per_touch"] = ((panel.prev_receiving_epa + panel.prev_rushing_epa)
                               / (panel.prev_targets + panel.prev_carries).clip(lower=1))
panel["first_round"] = (panel["round"] == 1).astype(int)

df = panel[panel.career_year.between(2, 4)].dropna(subset=["age"]).reset_index(drop=True)
print(f"{df.pfr_player_id.nunique()} players, {len(df)} player-seasons")
print(df.groupby("round").pfr_player_id.nunique().rename("players by round"))

### A helper for regressions (provided)

Each player appears up to three times (career years 2, 3 and 4), and his seasons are not independent of each other. `fit()` therefore **clusters standard errors by player**, the same idea as randomizing households instead of individuals in Quiz 4.

In [ ]:
def fit(formula, data, coef="first_round"):
    """OLS with standard errors clustered by player. Returns the coefficient on `coef`, its SE and 95% CI."""
    groups = data.pfr_player_id.astype("category").cat.codes
    r = smf.ols(formula, data).fit(cov_type="cluster", cov_kwds={"groups": groups})
    b, se = r.params[coef], r.bse[coef]
    return pd.Series({"estimate": b, "se": se, "ci_low": b - 1.96 * se, "ci_high": b + 1.96 * se,
                      "t": r.tvalues[coef], "n": int(r.nobs), "R2": r.rsquared})

## Part 1. The raw comparison (about 8 minutes)

**1a.** Compute the average `off_share` for first-round picks and for second- and third-round picks, and the difference.

In [ ]:
share_r1 = df.loc[df.first_round == 1, "off_share"].mean()
share_r23 = df.loc[df.first_round == 0, "off_share"].mean()
print(f"Round 1: {share_r1:.3f}   Rounds 2-3: {share_r23:.3f}   Difference: {share_r1 - share_r23:+.3f}")

**1b.** Now compare the two groups on their **previous-season** performance. (This is the same check as comparing the trained men with the CPS men in the LaLonde problem.)

In [ ]:
compare = ["prev_off_share", "prev_games", "prev_targets", "prev_receiving_yards", "prev_carries",
           "prev_rushing_yards", "prev_yds_per_target", "prev_epa_per_touch", "age"]
df.groupby("first_round")[compare].mean().T.rename(columns={0: "Rounds 2-3", 1: "Round 1"})

**1c.** Does the raw difference in 1a measure the sunk-cost effect?

✅ **Answer:** No. First-round picks play about **18 percentage points** more of their team's snaps. But the table shows they were also much more productive *last* season: more snaps, more targets, more yards, more efficient. Teams draft players in round 1 because they expect them to be better, and they usually are. So the raw gap mixes any sunk-cost effect with genuine talent. Talent is a **confounder**: it affects both draft position and playing time.

## Part 2. Adding controls (about 15 minutes)

We'll add controls in four groups, one at a time, and watch the first-round coefficient. This is the LaLonde exercise: which controls move the estimate most?

**2a.** Complete the list of cumulative formulas. Each formula should include *all* the control groups up to that point. (Hint: join strings with `" + "`.)

In [ ]:
groups = {
    "Position × career year, draft year": "C(position)*C(career_year) + C(draft_year)",
    "+ Prior-season efficiency":          "age + prev_yds_per_target + prev_yds_per_carry + prev_epa_per_touch "
                                          "+ prev_receiving_tds + prev_rushing_tds + prev_fumbles_lost_total",
    "+ Prior-season volume":              "prev_games + prev_targets + prev_carries + prev_receiving_yards + prev_rushing_yards",
    "+ Prior-season snap share":          "prev_off_share",
}

formulas = {"No controls": "off_share ~ first_round"}
controls = []
for name, terms in groups.items():
    controls.append(terms)
    formulas[name] = "off_share ~ first_round + " + " + ".join(controls)

results = pd.DataFrame({name: fit(f, df) for name, f in formulas.items()}).T
results

In [ ]:
FULL = formulas["+ Prior-season snap share"]   # we'll use the full specification from here on

ax = results.estimate.plot(kind="barh", xerr=1.96 * results.se, figsize=(7, 3), capsize=3)
ax.axvline(0, color="grey", lw=1); ax.invert_yaxis()
ax.set_xlabel("First-round coefficient on snap share (95% CI)")
plt.tight_layout(); plt.show()

**2b.** Which group of controls moves the estimate most, and why?

✅ **Answer:** Prior-season **efficiency** moves it most, cutting it roughly in half (from about 17 to about 9 points), and prior-season **volume** cuts it further to about 6. Once volume is included, prior snap share adds almost nothing, because volume already captures how much the player was used.

Previous-season performance matters because it is the best available measure of **talent**, the confounder from 1c. It plays the same role as **1974–75 earnings** in the LaLonde problem: one set of variables that summarizes how productive someone was *before* the comparison.

**2c.** Interpret the final coefficient.

✅ **Answer:** "Among receivers, backs and tight ends with the same production last season, first-round picks play about **6 percentage points more** of their team's offensive snaps, roughly 60–65 extra plays a season." The 95% confidence interval excludes zero (t ≈ 3), so it is statistically significant. That's consistent with a sunk-cost effect, *if* our controls capture talent, which Part 4 examines.

**2d. A bad control.** A classmate suggests also controlling for this season's **targets and carries**, since they're "good measures of performance". Run it.

In [ ]:
bad = FULL + " + targets + carries"   # current-season usage
pd.DataFrame({"Full controls": fit(FULL, df), "Plus current targets & carries": fit(bad, df)}).T

Why is this a bad idea?

✅ **Answer:** Current-season targets and carries are **outcomes of the coach's playing-time decision**. A player who is on the field more gets more targets and carries. Controlling for them asks "among players who got the same opportunities, did first-rounders play more?", which mostly strips out the very effect we want to measure. The estimate collapses toward zero.

This is the same mistake as comparing smokers' and nonsmokers' babies **only among low-birth-weight babies**: you condition on a variable that the treatment affects, so the comparison no longer measures the treatment's full effect and can even reverse sign. **Only control for variables determined before the treatment.**

## Part 3. Double machine learning (about 12 minutes)

Our regression assumes each control affects playing time **in a straight line and separately**. That's unlikely here. Yards per target means something different for a wide receiver than a tight end, and the effect of last season's snap share probably flattens out near 100%.

**Double ML** lets flexible prediction models handle the controls:

1. Predict **playing time** from the controls with a random forest. Keep the leftover (residual): actual minus predicted.
2. Predict **first-round status** from the same controls with another random forest. Keep its leftover.
3. Regress the playing-time leftover on the first-round leftover. The slope is the estimate.

**Cross-fitting:** each player's leftovers must come from models trained on *other* players. Otherwise the forests memorize the data and the leftovers are too small. We use `GroupKFold` so that **all of a player's seasons stay in the same fold**.

**3a.** Fill in the two TODO lines that compute the leftovers for the held-out fold.

In [ ]:
X = pd.get_dummies(df[["position"]], dtype=float)
X = pd.concat([X, df[["career_year", "draft_year", "age", "prev_off_share", "prev_games", "prev_targets",
                      "prev_receptions", "prev_receiving_yards", "prev_receiving_tds", "prev_receiving_epa",
                      "prev_carries", "prev_rushing_yards", "prev_rushing_tds", "prev_rushing_epa",
                      "prev_fumbles_lost_total", "prev_yds_per_target", "prev_yds_per_carry",
                      "prev_epa_per_touch"]]], axis=1).values
y = df.off_share.values
d = df.first_round.values.astype(float)

def forest():
    return RandomForestRegressor(n_estimators=500, min_samples_leaf=5, random_state=232, n_jobs=-1)

y_left, d_left = np.zeros(len(df)), np.zeros(len(df))
for train, test in GroupKFold(n_splits=5).split(X, y, groups=df.pfr_player_id):
    y_model = forest().fit(X[train], y[train])
    d_model = forest().fit(X[train], d[train])
    y_left[test] = y[test] - y_model.predict(X[test])
    d_left[test] = d[test] - d_model.predict(X[test])

dml = sm.OLS(y_left, d_left).fit(cov_type="cluster", cov_kwds={"groups": df.pfr_player_id.astype("category").cat.codes})
print(f"Double ML estimate: {dml.params[0]:.3f} (SE {dml.bse[0]:.3f})")
print(f"Share of playing-time variation explained by controls:   {1 - y_left.var() / y.var():.2f}")
print(f"Share of first-round variation explained by controls:    {1 - d_left.var() / d.var():.2f}")

**3b. The bug.** The cell below does the same thing **without cross-fitting**: each forest is trained on all the data and then predicts those same rows.

In [ ]:
y_fit = forest().fit(X, y).predict(X)
d_fit = forest().fit(X, d).predict(X)
no_cf = sm.OLS(y - y_fit, d - d_fit).fit()
print(f"Without cross-fitting: {no_cf.params[0]:.3f}")
print(f"In-sample share of playing-time variation 'explained': {1 - (y - y_fit).var() / y.var():.2f}")

**3c.** Compare the double ML estimate with the full regression.

✅ **Answer:** Double ML gives about **5.5 points**, almost the same as the regression's 5.6. Letting the forests find non-linear patterns and interactions in the controls didn't change the answer. So the regression wasn't badly misspecified, which is reassuring rather than disappointing. (The forests explain only a few percent of the variation in first-round status, which tells us performance alone can't predict round very well.)

**3d.** Why does the version without cross-fitting differ?

✅ **Answer:** Each forest has seen the rows it's predicting, so it partly **memorizes** them. The leftovers shrink, and the in-sample "explained" share is far higher than the honest out-of-sample one. Memorized leftovers no longer represent the variation the controls can't explain, which biases the estimate (here up to about 6.6). The forests claim to explain about 73% of the variation in playing time in-sample, against an honest 36% out-of-sample. This is overfitting, the same problem as training error vs. test error in the prediction unit.

Keeping each player's seasons in one fold prevents a subtler leak. A player's three seasons are very similar, so if two were in training and one in testing, the forest would effectively have seen him already.

**3e.** Does double ML solve unmeasured confounding?

✅ **Answer:** No. Double ML handles the **shape** of the relationship with the controls we *have*. It does nothing about controls we *don't* have, such as practice performance, coaches' private evaluations or off-field concerns. If those drive both draft position and playing time, double ML is just as biased as regression. Part 4 asks how big such a bias would need to be.

## Part 4. Sensitivity analysis (about 10 minutes)

The biggest worry is **unmeasured talent**: things coaches see that statistics miss. How strong would such a confounder need to be to wipe out our estimate?

The **robustness value** (Cinelli and Hazlett, 2020) answers this. It's the share of the *leftover* variation in both the treatment and the outcome that an unmeasured confounder would need to explain to bring the estimate to zero. With the regression's t-statistic and residual degrees of freedom:

$$f = \frac{|t|}{\sqrt{df}}, \qquad RV = \tfrac{1}{2}\left(\sqrt{f^4 + 4f^2} - f^2\right)$$

**4a.** Compute the robustness value for the full regression.

In [ ]:
full = smf.ols(FULL, df).fit()
t_stat, dof = full.tvalues["first_round"], full.df_resid

f = abs(t_stat) / math.sqrt(dof)
RV = 0.5 * (math.sqrt(f**4 + 4 * f**2) - f**2)
print(f"t = {t_stat:.2f}, df = {dof:.0f}, robustness value = {RV:.3f}")

**4b. Benchmarking.** Is that a lot? Compare it with how much our **strongest observed confounder**, the whole block of prior-season performance variables, explains. The cell below computes the block's *partial R²*: the share of the remaining variation it explains, once all the other controls are included, for both the outcome and the treatment.

In [ ]:
perf_terms = groups["+ Prior-season efficiency"] + " + " + groups["+ Prior-season volume"] + " + prev_off_share"
base = groups["Position × career year, draft year"]

def partial_r2(lhs, extra):
    """Share of remaining variation in `lhs` explained by the `perf_terms` block, given the other terms."""
    small = smf.ols(f"{lhs} ~ {extra} + {base}", df).fit().rsquared
    big = smf.ols(f"{lhs} ~ {extra} + {base} + {perf_terms}", df).fit().rsquared
    return (big - small) / (1 - small)

print(f"Prior-season performance explains {partial_r2('off_share', 'first_round'):.1%} of the remaining variation in playing time")
print(f"Prior-season performance explains {partial_r2('first_round', '1'):.1%} of the remaining variation in first-round status")

**4c.** What would an unmeasured confounder have to do, and how worried should we be?

✅ **Answer:** The robustness value is about **0.09**. An unmeasured factor, such as coaches' private view of a player's talent, would need to explain about **9% of the remaining variation in both** first-round status and playing time to bring the estimate to zero. (A weaker one could still make it statistically insignificant.)

Compare that with the benchmark. All the prior-season statistics together explain about **35%** of the remaining variation in playing time and about **12%** of the remaining variation in first-round status. Both are above 9%.

So an unmeasured factor **as strong as the prior-season statistics we already control for** would be enough to erase the estimate. That's not far-fetched: scouts see things statistics miss, such as speed, size and character, and coaches keep seeing them in practice. **The estimate is suggestive, not conclusive.** Sensitivity analysis doesn't tell us which story is true; it tells us exactly what we'd have to believe for the result to disappear.

## Part 5. Selecting on an outcome (about 7 minutes)

A classmate argues: *"Players who washed out of the league aren't relevant. Let's study only players still playing in their 4th season."*

**5a.** Restrict the data to players with `off_share > 0` in career year 4, and rerun the full regression.

In [ ]:
active_y4 = df.loc[(df.career_year == 4) & (df.off_share > 0), "pfr_player_id"].unique()
survivors = df[df.pfr_player_id.isin(active_y4)]

pd.DataFrame({"All players": fit(FULL, df), "Still playing in year 4": fit(FULL, survivors)}).T

**5b.** Which survivors are likely more talented in unmeasured ways, and why does the estimate move?

✅ **Answer:** The estimate falls from about 5.6 to about **3.6 points**, and it is no longer statistically significant. Second- and third-round picks only survive four years if they're genuinely good, because teams cut them quickly when they disappoint. First-rounders can survive while being mediocre, because teams give them extra chances. So **among survivors, the later picks are positively selected on unmeasured talent**. That talent earns them playing time, which makes the first-round advantage look smaller.

This is the birth-weight paradox. Survival here (like low birth weight there) is affected by **both** the treatment (draft round, like smoking) and an unmeasured factor (talent, like birth defects). Conditioning on it creates a link between the treatment and the unmeasured factor that doesn't exist in the full population, and biases the comparison. Here it biases the estimate toward zero.

(Players out of the league get 0 in our data, so the full sample doesn't have this problem, as long as we don't drop them.)

## Part 6. A regression discontinuity at the end of round 1 (about 15 minutes)

Talent changes smoothly with draft position: pick 32 and pick 33 should be about equally good. But pick 32 is a **first-round pick** and pick 33 is a **second-round pick**, which comes with a smaller contract and, from 2011 on, no fifth-year team option. If the "first-round" label itself raises playing time, we should see a **jump** in playing time at the cutoff.

The data below includes **every position** except kickers, punters and long snappers, drafted 2013–2021.
* `x` = pick number minus the cutoff (negative = round 1, positive = round 2)
* `early` = average share of snaps on the player's side of the ball in his first four seasons
* `team_prev_win` = the drafting team's win percentage in the previous season

In [ ]:
fix = {"SD": "LAC", "STL": "LA", "OAK": "LV"}   # old team codes in the schedule data
pfr_codes = {"NWE": "NE", "GNB": "GB", "KAN": "KC", "TAM": "TB", "NOR": "NO", "SFO": "SF",
             "SDG": "LAC", "STL": "LA", "LAR": "LA", "OAK": "LV", "LVR": "LV"}

reg = schedules[(schedules.game_type == "REG") & schedules.result.notna()]
wins = pd.concat([
    pd.DataFrame({"season": reg.season, "team": reg.home_team, "win": np.sign(reg.result).map({1: 1, 0: .5, -1: 0})}),
    pd.DataFrame({"season": reg.season, "team": reg.away_team, "win": np.sign(-reg.result).map({1: 1, 0: .5, -1: 0})})])
wins["team"] = wins.team.replace(fix)
wins = wins.groupby(["season", "team"]).win.mean().reset_index()
wins["season"] += 1   # previous season's record, matched to the draft year

rd = draft[draft.season.between(2013, 2021) & ~draft.position.isin(["K", "P", "LS"])].copy()
last_r1 = rd[rd["round"] == 1].groupby("season").pick.max()
rd["x"] = rd.pick - rd.season.map(last_r1) - 0.5   # round 1 had 31 picks in 2016 (New England forfeited one)
rd["first_round"] = (rd["round"] == 1).astype(int)
for k in range(1, 5):
    s = player_season[["pfr_player_id", "season", "play_share"]].copy()
    s["season"] -= k - 1
    rd = rd.merge(s.rename(columns={"play_share": f"yr{k}"}), on=["pfr_player_id", "season"], how="left")
rd["early"] = rd[["yr1", "yr2", "yr3", "yr4"]].fillna(0).mean(axis=1)
rd["tm"] = rd.team.replace(pfr_codes)
rd = rd.merge(wins.rename(columns={"team": "tm", "win": "team_prev_win"}), on=["season", "tm"], how="left")
rd["qb"] = (rd.position == "QB").astype(int)

near = rd[rd.x.abs() < 16]   # bandwidth: 16 picks either side
binned = near.groupby("x")[["early", "team_prev_win"]].mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, col, label in [(axes[0], "early", "Share of snaps, years 1-4"),
                       (axes[1], "team_prev_win", "Drafting team's previous win %")]:
    ax.scatter(binned.index, binned[col], s=15)
    for side in (near.x < 0, near.x > 0):
        b = np.polyfit(near.x[side], near[col][side], 1)
        xs = np.array([near.x[side].min(), near.x[side].max()])
        ax.plot(xs, b[0] * xs + b[1])
    ax.axvline(0, color="grey", ls="--", lw=1); ax.set_xlabel("Picks from the end of round 1"); ax.set_title(label)
plt.tight_layout(); plt.show()

**6a.** Estimate the jump in early-career playing time at the cutoff with a piecewise linear regression, as in the tariff problem: `early` on `first_round`, `x` and their interaction. Use robust standard errors (`cov_type="HC1"`). Try bandwidths of 8, 12, 16 and 24 picks.

In [ ]:
rows = {}
for h in [8, 12, 16, 24]:
    w = rd[rd.x.abs() < h]
    r = smf.ols("early ~ first_round * x", w).fit(cov_type="HC1")
    rows[h] = {"jump": r.params["first_round"], "se": r.bse["first_round"], "n": len(w)}
pd.DataFrame(rows).T.rename_axis("bandwidth (picks)")

**6b.** The sunk-cost story predicts a jump *up* for first-round picks. What do you find?

✏️ **Your answer:**

&nbsp;

**6c. Checking the requirements.** Repeat 6a with `team_prev_win` and `qb` as the outcome, at a bandwidth of 16. These are characteristics fixed before the pick, like the covariate checks in RDD.

In [ ]:
w = rd[rd.x.abs() < 16]
pd.DataFrame({out: smf.ols(f"{out} ~ first_round * x", w).fit(cov_type="HC1").params[["first_round"]]
              for out in ["team_prev_win", "qb"]}).T

In [ ]:
# Who was picked right at the cutoff?
rd[rd.x.abs() < 1][["season", "pick", "pfr_player_name", "position", "team", "team_prev_win"]]

**6b.** What do you find?

✅ **Answer:** The opposite of the sunk-cost prediction. Late first-round picks play **less** than early second-round picks: about 14–21 percentage points less of their side's snaps over their first four seasons, and the jump is statistically significant at all four bandwidths. Taken at face value, the round-1 label *reduces* playing time.

**6c / 6d.** Explain the jump in `team_prev_win`. Which requirement fails?

✅ **Answer:** Because draft order restarts each round, **pick 32 belongs to the Super Bowl champion and pick 33 to the worst team in the league**. The drafting team's previous win percentage jumps by about **40 percentage points** at the cutoff. Two things change at once: the player's round *and* the quality of the team he joins. (The share of quarterbacks is about 4 points higher just inside round 1, but that isn't statistically significant.)

That violates the **single change** requirement, like the steel tariffs that took effect alongside the washing-machine tariffs. A late first-rounder joins a strong team with established starters ahead of him. An early second-rounder joins a weak team where he can start right away. That alone could explain the drop in 6b, and it could hide a real sunk-cost effect in the other direction. Adding `team_prev_win` as a control doesn't fix it, because team quality jumps almost exactly at the cutoff. This RDD can't separate the label from the team.

**6e.** Which requirement do trade-ups threaten?

✅ **Answer:** **No sorting.** Teddy Bridgewater (2014) and Lamar Jackson (2018) were both quarterbacks taken at pick 32 by teams that traded up, reportedly to get the extra contract year. If teams deliberately place certain players just inside the cutoff, players on either side are no longer comparable. Here the main problem is still 6d; sorting adds a second reason for caution.

## Part 7. Bringing it together (about 5 minutes)

**7.** Advise the budget office.

✅ **Answer (example):** Comparing projects with similar performance reviews is a sensible start, because the raw cost–funding relationship mostly reflects that expensive projects were expected to be more valuable. Our controls cut the NFL estimate by two thirds. But reviews will miss things decision-makers know, such as political priorities, expected future benefits or a project manager's reputation. Those may drive both cost and continued funding, just as scouts' private views of talent might here, so report a sensitivity analysis showing how strong such a factor would need to be.

Don't control for anything decided after the funding choice, such as this year's spending or staffing. And don't restrict the sample to projects that survived, since survival itself depends on cost. A natural cutoff, such as a dollar threshold that triggers extra review, could support a regression discontinuity, but first check that nothing else changes at the threshold.